# Irrigation Optimization — Reinforcement Learning
## Problem Definition & MDP Formulation

**Author:** [Your Name]
**Project:** AI-Powered Agricultural Mobile Application
**Crop:** Wheat (*Triticum aestivum*), Tunisia
**Season:** November → June (~228 days)

---

## 1. Business Objective

Design an autonomous irrigation system that:

- Maintains root-zone soil moisture within the optimal range for wheat growth
- Minimizes total water consumption per season
- Prevents crop water stress without over-irrigating
- Generalizes across multiple growing seasons (2022–2026)

## 2. Data Science Objective

Develop a **Reinforcement Learning (RL) agent** that learns an optimal  
daily irrigation policy by interacting with a **learned soil moisture simulator**.

The system is built in two connected layers:

| Layer | Role | Method |
|-------|------|--------|
| **Regression Simulator** | Learns how soil moisture responds to irrigation, weather, and ETc | XGBoost / Random Forest |
| **RL Controller** | Learns the optimal irrigation decision given the current state | PPO / DQN |

The regression simulator acts as a **digital twin of the farm** — the RL agent  
trains inside it rather than on a real field, which would be destructive and slow.

This architecture is known as **Model-Based Reinforcement Learning (MBRL)**.

## 3. Reinforcement Learning Formulation

Irrigation scheduling is modeled as a **Markov Decision Process (MDP)**,  
defined by the tuple **(S, A, R, T, γ)**:

| Component | Symbol | Description |
|-----------|--------|-------------|
| State space | S | What the agent observes each day |
| Action space | A | How much to irrigate |
| Reward function | R | Score given after each decision |
| Transition function | T | How the environment responds (the simulator) |
| Discount factor | γ | How much future rewards are valued |
| Episode | — | One full wheat season (sowing → harvest) |

## 4. State Space (S)

At each day **t**, the agent observes a state vector **S_t** containing  
the current environmental and crop conditions.

### State vector

```
S_t = [
    sm_root_t,        # Root-zone soil moisture (m³/m³) — weighted avg of 9–27cm & 27–81cm
    precip_mm_t,      # Precipitation today (mm)
    et0_mm_t,         # Reference evapotranspiration ET0 (mm/day)
    etc_mm_t,         # Crop evapotranspiration ETc = ET0 × Kc (mm/day)
    kc_t,             # Crop coefficient (FAO-56, stage-dependent)
    temp_c_t,         # Air temperature (°C)
    rh_pct_t,         # Relative humidity (%)
    wind_kmh_t,       # Wind speed (km/h)
    day_of_season_t   # Day index within current season (1–228)
]
```

### Design decisions

**Why `sm_root` instead of two separate SM layers?**  
The two raw ERA5 layers (9–27cm and 27–81cm) are combined into a single  
root-zone variable. The agent only needs one moisture signal — the state  
would be redundant and harder to learn from with two correlated variables.

```
sm_root = 0.6 × sm_9_27cm + 0.4 × sm_27_81cm
```

**Why include temperature, humidity and wind?**  
These variables drive evapotranspiration. Without them the agent cannot  
anticipate how fast the soil will dry out in the coming hours — it would  
be making decisions blind to the atmospheric demand.

**Why `day_of_season`?**  
Crop water requirements change significantly across the season (low Kc at  
germination, peak Kc at mid-season). The agent needs to know which stage  
it is at to calibrate its irrigation decisions appropriately.

## 5. Action Space (A)

The agent decides **how much water to apply** each day.

### Continuous action space

```
A_t ∈ [0, 20]  mm/day
```

A **continuous** action space is used rather than discrete levels  
(0, 5, 10, 15, 20 mm). This allows the agent to find precise irrigation  
amounts rather than being constrained to fixed steps.

The upper bound of 20 mm/day reflects a realistic physical constraint  
for drip/sprinkler irrigation systems used in Tunisian wheat farming.

### Algorithm implication

Continuous actions require an actor-critic algorithm such as:
- **PPO** (Proximal Policy Optimization) — recommended for stability
- **SAC** (Soft Actor-Critic) — recommended for sample efficiency

Both support continuous action spaces natively.

## 6. Transition Function (T) — The Regression Simulator

The transition function answers:
> *"Given today's state and the irrigation decision, what will soil moisture be tomorrow?"*

### Architecture

Rather than using a fixed physical water balance equation, we train a  
**regression model** (XGBoost or Random Forest) on the prepared dataset  
to learn the soil moisture dynamics from data.

The model predicts the **daily change** in root-zone moisture:

```
delta_sm = f(sm_root, precip_mm, et0_mm, kc, temp_c, irrigation_mm, net_balance_mm)
```

Then the next state is computed as:

```
sm_root(t+1) = sm_root(t) + delta_sm
sm_root(t+1) = clip(sm_root(t+1), WP, FC)    # physical bounds
```

### Why a learned simulator instead of a fixed equation?

A fixed water balance equation:
```
SM(t+1) = SM(t) + precipitation + irrigation − ETc
```
assumes a perfectly known, linear system. In reality, soil moisture  
dynamics are non-linear — drainage, capillary rise, and spatial  
heterogeneity all affect the true response. A learned simulator  
captures these effects directly from the observed data.

### Soil parameters (derived from data)

Since no soil lab report was available, parameters were estimated  
from the statistical distribution of observed `sm_root`:

| Parameter | Value | Derivation |
|-----------|-------|------------|
| Field Capacity (FC) | 95th percentile of sm_root | Upper bound soil can hold |
| Wilting Point (WP) | 5th percentile of sm_root | Lower safe moisture bound |
| Available Water (AW) | FC − WP | Usable moisture range |
| MAD | 40% | FAO-56 standard for wheat |
| Stress Threshold | FC − (MAD × AW) | Irrigation trigger point |

## 7. Reward Function (R)

The reward function guides the agent toward the dual objective:  
**minimize water use** while **preventing crop stress**.

### Formulation

```
R_t = - α × irrigation_mm_t          # penalize water use
      - β × stress_penalty_t          # penalize going below threshold
      - γ × overwater_penalty_t       # penalize exceeding FC
```

Where the penalty terms are defined as:

```python
# Stress penalty: how far below the stress threshold is the soil?
stress_penalty    = max(0, THRESHOLD - sm_root_t)

# Overwatering penalty: how far above field capacity is the soil?
overwater_penalty = max(0, sm_root_t - FC)
```

### Penalty weights

| Weight | Value | Rationale |
|--------|-------|-----------|
| α (water cost) | 0.1 | Small penalty per mm — water saving is secondary to yield |
| β (stress penalty) | 10.0 | Large penalty — crop stress directly reduces yield |
| γ (overwater penalty) | 5.0 | Medium penalty — waterlogging damages roots |

The asymmetry between β and γ reflects the agricultural reality:  
under-irrigation is more damaging to wheat than mild over-irrigation.

### Reward signal behaviour

| Situation | Reward |
|-----------|--------|
| Soil in optimal range, no irrigation | 0 (best possible) |
| Correct irrigation to avoid stress | Small negative (−water cost) |
| Soil below threshold, no irrigation | Large negative (−β × deficit) |
| Excessive irrigation above FC | Medium negative (−γ × excess) |

## 8. Episode Structure

One episode corresponds to **one complete wheat season**  
from sowing (November 1) to harvest (June 15) — approximately 228 days.

```
Episode start  → Day 1  (Sowing,   Nov 1)
                  Day 20  (End of initial stage,  Kc = 0.70)
                  Day 50  (End of development,    Kc = 1.15)
                  Day 100 (End of mid-season,     Kc = 1.15)
                  Day 175 (End of late-season,    Kc = 0.35)
Episode end    → Day 228 (Harvest, Jun 15)
```

### Training across multiple seasons

The dataset spans **4 seasons (2022–2026)**. During training, the agent  
cycles through all seasons as separate episodes. This exposes it to the  
natural inter-annual climate variability in Tunisia — dry years, wet years,  
hot springs — making the learned policy more robust.

## 9. Full Pipeline

```
Raw Data (weather.csv + soil_moisture.csv)
            │
            ▼
    Data Preparation
    ─────────────────────────────────────────────
    • Merge & aggregate to daily resolution
    • Filter to wheat crop season (4 seasons)
    • Compute Kc (FAO-56, 228-day curve)
    • Create sm_root (weighted SM average)
    • Estimate FC, WP, stress threshold from data
    • Generate synthetic irrigation (FAO deficit rule)
    • Compute delta_sm (regression target)
            │
            ▼
    Regression Simulator Training
    ─────────────────────────────────────────────
    Input  : sm_root, precip, et0, kc, temp,
             irrigation_mm, net_balance_mm
    Target : delta_sm
    Model  : XGBoost / Random Forest
            │
            ▼
    RL Agent Training (inside simulator)
    ─────────────────────────────────────────────
    State  : [sm_root, precip, et0, etc, kc,
              temp, rh, wind, day_of_season]
    Action : irrigation_mm ∈ [0, 20]
    Reward : −water − stress − overwater
    Algo   : PPO (continuous action)
            │
            ▼
    Deployment
    ─────────────────────────────────────────────
    Daily: observe state → RL outputs irrigation (mm)
    App shows: recommended irrigation + expected SM tomorrow
```

## 10. Next Steps

| Notebook | Description |
|----------|-------------|
| `02_data_preparation.ipynb` | Raw data → clean RL-ready dataset |
| `03_simulator_training.ipynb` | Train regression simulator (delta_sm model) |
| `04_rl_environment.ipynb` | Build custom OpenAI Gym environment using the simulator |
| `05_rl_training.ipynb` | Train PPO agent, evaluate irrigation policy |
| `06_evaluation.ipynb` | Compare RL policy vs baseline (fixed schedule, no irrigation) |

---

*Reference: Allen, R.G., Pereira, L.S., Raes, D., Smith, M. (1998).  
Crop evapotranspiration — FAO Irrigation and Drainage Paper 56. FAO, Rome.*